# Amazon Reviews 2023 — Electronics EDA

Perform exploratory data analysis (EDA) on the Amazon reviews dataset (Lesson 8):

- Distribution of ratings
- Correlation between ratings and price
- Popular items / categories by review count

Dataset: https://amazon-reviews-2023.github.io/ — files in `data/raw/`:
`Electronics.jsonl.gz` (~6 GB, reviews) and `meta_Electronics.jsonl.gz` (~1.2 GB, item metadata), joined on `parent_asin`.

**Approach:** the review file is far too large to load into pandas, so we use **DuckDB** to scan the gzipped JSONL out-of-core. We first do a one-off **ETL pass** projecting only the columns we need into compact Parquet under `data/processed/`; every analysis below then runs fast on Parquet.

In [ ]:
import os, time
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("../../").resolve()          # notebook lives in notebooks/week1/
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
(PROC / "_tmp").mkdir(parents=True, exist_ok=True)

REVIEWS_GZ = RAW / "Electronics.jsonl.gz"
META_GZ    = RAW / "meta_Electronics.jsonl.gz"
REVIEWS_PQ = PROC / "reviews.parquet"
META_PQ    = PROC / "meta.parquet"

con = duckdb.connect()
con.execute(f"SET memory_limit='6GB'; SET threads={os.cpu_count()}; "
            f"SET temp_directory='{PROC / '_tmp'}';")

## ETL: gzipped JSONL → compact Parquet
Scans each `.gz` once (reviews ~30s, meta ~10s). Idempotent — skips if the Parquet already exists.

In [ ]:
def etl():
    if not REVIEWS_PQ.exists():
        t = time.time()
        con.execute(f"""
          COPY (
            SELECT rating, parent_asin
            FROM read_json('{REVIEWS_GZ}',
                 columns={{'rating':'DOUBLE','parent_asin':'VARCHAR'}}, ignore_errors=true)
            WHERE parent_asin IS NOT NULL
          ) TO '{REVIEWS_PQ}' (FORMAT parquet);""")
        print(f"wrote reviews.parquet in {time.time()-t:.0f}s")
    if not META_PQ.exists():
        t = time.time()
        con.execute(f"""
          COPY (
            SELECT parent_asin,
                   TRY_CAST(price AS DOUBLE) AS price,   -- '19.99' -> 19.99, '—' -> NULL
                   main_category, title, categories
            FROM read_json('{META_GZ}',
                 columns={{'parent_asin':'VARCHAR','price':'VARCHAR','main_category':'VARCHAR',
                           'title':'VARCHAR','categories':'VARCHAR[]'}}, ignore_errors=true)
            WHERE parent_asin IS NOT NULL
          ) TO '{META_PQ}' (FORMAT parquet);""")
        print(f"wrote meta.parquet in {time.time()-t:.0f}s")

etl()
n_rev  = con.sql(f"SELECT COUNT(*) FROM '{REVIEWS_PQ}'").fetchone()[0]
n_meta = con.sql(f"SELECT COUNT(*) FROM '{META_PQ}'").fetchone()[0]
print(f"reviews = {n_rev:,}   items (meta) = {n_meta:,}")

## 1. Distribution of ratings

In [ ]:
dist = con.sql(f"""SELECT rating, COUNT(*) AS n FROM '{REVIEWS_PQ}'
                   WHERE rating BETWEEN 1 AND 5 GROUP BY rating ORDER BY rating""").df()
dist["pct"] = dist.n / dist.n.sum() * 100
mean_rating = con.sql(f"SELECT AVG(rating) FROM '{REVIEWS_PQ}' "
                      f"WHERE rating BETWEEN 1 AND 5").fetchone()[0]

plt.figure(figsize=(6,4))
plt.bar(dist.rating, dist.n, width=0.6, color="#4C78A8")
plt.xlabel("rating"); plt.ylabel("# reviews"); plt.title(f"Rating distribution (mean={mean_rating:.2f})")
plt.tight_layout(); plt.show()
dist

Ratings are **heavily left-skewed**: ~63% are 5-star and ~83% are 4–5 stars, with a small bump at 1-star — the classic J-shaped review distribution. Mean ≈ **4.10**.

## Item-level table
Aggregate reviews per `parent_asin`, then join to metadata (price, title, category).

In [ ]:
items = con.sql(f"""
  WITH r AS (SELECT parent_asin, COUNT(*) AS n_reviews, AVG(rating) AS avg_rating
             FROM '{REVIEWS_PQ}' GROUP BY parent_asin)
  SELECT r.parent_asin, r.n_reviews, r.avg_rating,
         m.price, m.title, m.main_category, m.categories
  FROM r LEFT JOIN '{META_PQ}' m USING (parent_asin)
""").df()
print(f"items = {len(items):,}   with price = {items.price.notna().sum():,} "
      f"({items.price.notna().mean():.0%})")
items.head()

## 2. Correlation between ratings and price
At the item level (avg rating vs price), trimming prices above the 99th percentile to remove outliers.

In [ ]:
pv = items.dropna(subset=["price"]).query("price > 0")
pv = pv[pv.price <= pv.price.quantile(0.99)].copy()

pearson  = pv["avg_rating"].corr(pv["price"])
spearman = pv["avg_rating"].rank().corr(pv["price"].rank())  # = Spearman, no scipy needed
print(f"items used = {len(pv):,}   Pearson = {pearson:.3f}   Spearman = {spearman:.3f}")

pv["price_decile"] = pd.qcut(pv.price, 10, duplicates="drop")
binned = (pv.groupby("price_decile", observed=True)
            .agg(avg_rating=("avg_rating","mean"),
                 price_mid=("price","median"),
                 n_items=("avg_rating","size")).reset_index())

plt.figure(figsize=(6,4))
plt.plot(binned.price_mid, binned.avg_rating, "o-", color="#E45756")
plt.xlabel("price (decile median, $)"); plt.ylabel("mean item avg_rating")
plt.title("Avg rating by price decile"); plt.tight_layout(); plt.show()
binned[["price_mid","avg_rating","n_items"]]

**Essentially no linear correlation** between price and rating (Pearson ≈ -0.02, Spearman ≈ -0.04). Average rating is flat (~4.05) across price deciles, dipping only slightly for the most expensive items. Price is not a predictor of satisfaction here. (Note: only ~33% of items carry a price in the metadata.)

## 3a. Most popular items by review count

In [ ]:
top_items = items.sort_values("n_reviews", ascending=False).head(15)
top_items[["title","n_reviews","avg_rating","price"]]

The most-reviewed items are dominated by **Amazon's own devices** (Fire TV Stick, Echo Dot, Fire tablets), followed by popular budget earbuds (TOZO, Panasonic).

## 3b. Most popular categories by review count

In [ ]:
cats = con.sql(f"""
  WITH r AS (SELECT parent_asin, COUNT(*) AS n FROM '{REVIEWS_PQ}' GROUP BY parent_asin)
  SELECT m.categories[2] AS subcategory, SUM(r.n) AS reviews, COUNT(*) AS items
  FROM r JOIN '{META_PQ}' m USING (parent_asin)
  WHERE len(m.categories) >= 2 AND m.categories[2] IS NOT NULL
  GROUP BY subcategory ORDER BY reviews DESC LIMIT 15
""").df()

plt.figure(figsize=(7,5))
plt.barh(cats.subcategory[::-1], cats.reviews[::-1], color="#54A24B")
plt.xlabel("# reviews"); plt.title("Top sub-categories by reviews")
plt.tight_layout(); plt.show()
cats

## Summary of findings

- **Scale:** ~43.9M reviews across ~1.6M items.
- **Ratings:** strongly positive / J-shaped — ~63% 5-star, mean ≈ 4.10.
- **Ratings vs price:** no meaningful correlation (Pearson ≈ -0.02); ratings are flat across price levels.
- **Popular items:** Amazon first-party devices (Fire TV Stick, Echo Dot) dominate review volume.
- **Popular categories:** *Computers & Accessories* is the largest sub-category by reviews (~17.6M), then *Camera & Photo* and *Headphones/Earbuds*.
- **Data quality:** only ~33% of items have a price in metadata, so price-based analysis covers a subset.